#BATCH 10 --> LOAN APPROVAL EXPERT SYSTEM
##ARTIFICIAL INTELLIGENCE (MAS5001)
**Siddhartha Parida (25MAS10023),
Anirudh Narayanan (25MAS10011)**

EXPERT SYSTEM MODEL

In [ ]:
class LoanApprovalExpertSystem:
    def __init__(self, rules):
        #Initializes the expert system with a set of rules.
        self.rules = rules

    def _check_condition(self, condition, facts):
        # Checks if a single condition is met by the facts.
        fact_name, operator, value = condition
        applicant_value = facts.get(fact_name)

        if applicant_value is None:
            return False

        # Handle the specific case where the value is a fact name to compare against
        if isinstance(value, str) and value in facts:
            value_to_compare = facts.get(value)
        else:
            value_to_compare = value

        if operator == '>': return applicant_value > value_to_compare
        if operator == '<': return applicant_value < value_to_compare
        if operator == '>=': return applicant_value >= value_to_compare
        if operator == '<=': return applicant_value <= value_to_compare
        if operator == '==': return applicant_value == value_to_compare
        if operator == '!=': return applicant_value != value_to_compare
        return False

    def run(self, facts):
        # Running the facts agains the KB
        print("Inialising Loan Application Review...")
        print(f"Applicant Profile: {facts}")

        # Initializing decision score and reasons
        facts['decision_score'] = 0
        facts['reasons'] = []

        # Forward-chaining inference engine
        for rule in self.rules:
            conditions = rule['if']

            # Checking conditions of reules to be met
            all_conditions_met = all(self._check_condition(c, facts) for c in conditions)

            if all_conditions_met:
                action = rule['then']
                action_type, param, value = action

                print(f"Rule Check: {rule['name']}")

                if action_type == 'adjust_score':
                    facts['decision_score'] += value
                elif action_type == 'add_reason':
                    facts['reasons'].append(value)

        # Final decision on the basis of score
        final_score = facts['decision_score']
        print(f"\nFinal Score Calculation -->")
        print(f"Total Decision Score: {final_score}")

        if final_score >= 10:
            decision = f"{facts['name']} has been Approved!"
            facts['reasons'].append(f"Congratulations {facts['name']}! Your profile meets the requirements for loan.")
        elif 0 <= final_score < 10:
            decision = f"{facts['name']} Requires Manual review!"
            facts['reasons'].append(f"Sorry {facts['name']}! Your profile is borderline and requires review by a loan officer.")
        else:
            decision = f"{facts['name']} has been Declined!"
            facts['reasons'].append(f"Sorry {facts['name']}! Your profile does not meet the minimum requirements for a loan.")

        return decision, facts['reasons']

# The Knowledge Base(KB) with 50 Rules
KNOWLEDGE_BASE = [
    # Credit Score Rules (10)
    {'name': 'Excellent Credit Score', 'if': [('credit_score', '>=', 800)], 'then': ('adjust_score', 'score', 20)},
    {'name': 'Very Good Credit Score', 'if': [('credit_score', '>=', 740), ('credit_score', '<', 800)], 'then': ('adjust_score', 'score', 15)},
    {'name': 'Good Credit Score', 'if': [('credit_score', '>=', 670), ('credit_score', '<', 740)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'Fair Credit Score', 'if': [('credit_score', '>=', 580), ('credit_score', '<', 670)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Poor Credit Score', 'if': [('credit_score', '<', 580)], 'then': ('adjust_score', 'score', -25)},
    {'name': 'Recent Bankruptcy', 'if': [('bankruptcy_in_last_7_years', '==', True)], 'then': ('adjust_score', 'score', -30)},
    {'name': 'No Recent Bankruptcy', 'if': [('bankruptcy_in_last_7_years', '==', False)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Multiple Recent Inquiries', 'if': [('credit_inquiries_last_2_years', '>', 5)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'History of Late Payments', 'if': [('late_payments_in_last_2_years', '>', 2)], 'then': ('adjust_score', 'score', -10)},
    {'name': 'No Late Payments', 'if': [('late_payments_in_last_2_years', '==', 0)], 'then': ('adjust_score', 'score', 5)},

    # Income & Employment Rules (20)
    {'name': 'Very High Income', 'if': [('annual_income', '>=', 150000)], 'then': ('adjust_score', 'score', 20)},
    {'name': 'High Income', 'if': [('annual_income', '>=', 75000), ('annual_income', '<', 150000)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'Moderate Income', 'if': [('annual_income', '>=', 40000), ('annual_income', '<', 75000)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Low Income', 'if': [('annual_income', '<', 40000)], 'then': ('adjust_score', 'score', -10)},
    {'name': 'Stable Employment', 'if': [('years_at_current_job', '>=', 5)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'Reasonably Stable Employment', 'if': [('years_at_current_job', '>=', 2), ('years_at_current_job', '<', 5)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'New Employment', 'if': [('years_at_current_job', '<', 2)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Unemployed Applicant', 'if': [('employment_status', '==', 'Unemployed')], 'then': ('adjust_score', 'score', -40)},
    {'name': 'Self-Employed High Earner', 'if': [('employment_status', '==', 'Self-Employed'), ('annual_income', '>=', 100000)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Self-Employed Low Earner', 'if': [('employment_status', '==', 'Self-Employed'), ('annual_income', '<', 100000)], 'then': ('adjust_score', 'score', -5)},

    # Debt & Financial Ratios (30)
    {'name': 'Very Low DTI Ratio', 'if': [('debt_to_income_ratio', '<', 0.20)], 'then': ('adjust_score', 'score', 15)},
    {'name': 'Low DTI Ratio', 'if': [('debt_to_income_ratio', '>=', 0.20), ('debt_to_income_ratio', '<', 0.36)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'Acceptable DTI Ratio', 'if': [('debt_to_income_ratio', '>=', 0.36), ('debt_to_income_ratio', '<=', 0.43)], 'then': ('adjust_score', 'score', 0)},
    {'name': 'High DTI Ratio', 'if': [('debt_to_income_ratio', '>', 0.43)], 'then': ('adjust_score', 'score', -15)},
    {'name': 'No Existing Loans', 'if': [('has_existing_loan', '==', False)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Has Existing Loan', 'if': [('has_existing_loan', '==', True)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Large Savings', 'if': [('liquid_savings', '>=', 50000)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'Moderate Savings', 'if': [('liquid_savings', '>=', 10000), ('liquid_savings', '<', 50000)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Low Savings', 'if': [('liquid_savings', '<', 10000)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'High Credit Utilization', 'if': [('credit_card_utilization', '>', 0.5)], 'then': ('adjust_score', 'score', -10)},

    # Loan Details (40)
    {'name': 'Small Loan Amount', 'if': [('loan_amount', '<=', 10000)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Large Loan Amount', 'if': [('loan_amount', '>', 250000)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Debt Consolidation Purpose', 'if': [('loan_purpose', '==', 'Debt Consolidation')], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Home Improvement Purpose', 'if': [('loan_purpose', '==', 'Home Improvement')], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Business Loan Purpose', 'if': [('loan_purpose', '==', 'Business')], 'then': ('adjust_score', 'score', -10)},
    {'name': 'Low Loan-to-Value (Mortgage)', 'if': [('loan_to_value_ratio', '<', 0.8)], 'then': ('adjust_score', 'score', 10)},
    {'name': 'High Loan-to-Value (Mortgage)', 'if': [('loan_to_value_ratio', '>=', 0.95)], 'then': ('adjust_score', 'score', -10)},
    {'name': 'Short Loan Term', 'if': [('loan_term_months', '<=', 36)], 'then': ('adjust_score', 'score', 5)},
    {'name': 'Long Loan Term', 'if': [('loan_term_months', '>', 60)], 'then': ('adjust_score', 'score', -5)},
    {'name': 'Loan Amount Exceeds Income', 'if': [('loan_amount', '>', 'annual_income')], 'then': ('adjust_score', 'score', -20)},

    # Combined Risk Factors (50)
    {'name': 'Risk: Low Income & High DTI', 'if': [('annual_income', '<', 40000), ('debt_to_income_ratio', '>', 0.4)], 'then': ('adjust_score', 'score', -20)},
    {'name': 'Risk: Poor Credit & New Job', 'if': [('credit_score', '<', 600), ('years_at_current_job', '<', 1)], 'then': ('adjust_score', 'score', -20)},
    {'name': 'Strength: High Income & Low DTI', 'if': [('annual_income', '>', 100000), ('debt_to_income_ratio', '<', 0.2)], 'then': ('adjust_score', 'score', 15)},
    {'name': 'Strength: Excellent Credit & Stable Job', 'if': [('credit_score', '>', 750), ('years_at_current_job', '>', 5)], 'then': ('adjust_score', 'score', 15)},
    {'name': 'Risk: Self-Employed & High DTI', 'if': [('employment_status', '==', 'Self-Employed'), ('debt_to_income_ratio', '>', 0.4)], 'then': ('adjust_score', 'score', -10)},
    {'name': 'Risk: Large Loan & Low Savings', 'if': [('loan_amount', '>', 50000), ('liquid_savings', '<', 5000)], 'then': ('add_reason', 'reason', 'Insufficient savings for the requested loan amount.')},
    {'name': 'Risk: Past Bankruptcy & High DTI', 'if': [('bankruptcy_in_last_7_years', '==', True), ('debt_to_income_ratio', '>', 0.4)], 'then': ('adjust_score', 'score', -15)},
    {'name': 'Applicant Too Young', 'if': [('age', '<', 18)], 'then': ('adjust_score', 'score', -100)},
    {'name': 'Non-Citizen Risk', 'if': [('is_citizen', '==', False)], 'then': ('add_reason', 'reason', 'Non-citizen status requires special review.')},
    {'name': 'Very High Risk Applicant', 'if': [('credit_score', '<', 500), ('annual_income', '<', 20000)], 'then': ('adjust_score', 'score', -50)},
]

INPUT OF CUSTOMER DETAILS AND OUTPUT

In [ ]:
if __name__ == "__main__":
    # Creating an instance of the expert system
    loan_expert = LoanApprovalExpertSystem(KNOWLEDGE_BASE)

    """
      3 different applicant cases
    """
    # Applicant 1: Strong Candidate
    applicant_1 = {
        'name': 'John Doe',
        'credit_score': 810,
        'annual_income': 160000,
        'debt_to_income_ratio': 0.15,
        'years_at_current_job': 8,
        'employment_status': 'Employed',
        'bankruptcy_in_last_7_years': False,
        'late_payments_in_last_2_years': 0,
        'credit_inquiries_last_2_years': 1,
        'has_existing_loan': False,
        'liquid_savings': 75000,
        'loan_amount': 25000,
        'loan_purpose': 'Home Improvement',
        'loan_to_value_ratio': 0.5,
        'loan_term_months': 36,
        'age': 45,
        'is_citizen': True
    }
    decision_1, reasons_1 = loan_expert.run(applicant_1)
    print(f"\nFINAL DECISION: {decision_1}")
    print(f"Reasons: {reasons_1}\n\n" )

    # Applicant 2: Weak Candidate
    applicant_2 = {
        'name': 'Tasha Blake',
        'credit_score': 550,
        'annual_income': 35000,
        'debt_to_income_ratio': 0.55,
        'years_at_current_job': 1,
        'employment_status': 'Employed',
        'bankruptcy_in_last_7_years': True,
        'late_payments_in_last_2_years': 4,
        'credit_inquiries_last_2_years': 7,
        'has_existing_loan': True,
        'liquid_savings': 1000,
        'loan_amount': 40000,
        'loan_purpose': 'Debt Consolidation',
        'loan_to_value_ratio': 0.98,
        'loan_term_months': 72,
        'age': 28,
        'is_citizen': True
    }
    decision_2, reasons_2 = loan_expert.run(applicant_2)
    print(f"\nFINAL DECISION: {decision_2}")
    print(f"Reasons: {reasons_2}\n\n")

    # Applicant 3: Borderline Candidate
    applicant_3 = {
        'name': 'Sid Par',
        'credit_score': 680,
        'annual_income': 60000,
        'debt_to_income_ratio': 0.42,
        'years_at_current_job': 3,
        'employment_status': 'Self-Employed',
        'bankruptcy_in_last_7_years': False,
        'late_payments_in_last_2_years': 1,
        'credit_inquiries_last_2_years': 2,
        'has_existing_loan': True,
        'liquid_savings': 12000,
        'loan_amount': 20000,
        'loan_purpose': 'Business',
        'loan_to_value_ratio': 0.85,
        'loan_term_months': 60,
        'age': 35,
        'is_citizen': True
    }
    decision_3, reasons_3 = loan_expert.run(applicant_3)
    print(f"\nFINAL DECISION: {decision_3}")
    print(f"Reasons: {reasons_3}\n\n")

Inialising Loan Application Review...
Applicant Profile: {'name': 'John Doe', 'credit_score': 810, 'annual_income': 160000, 'debt_to_income_ratio': 0.15, 'years_at_current_job': 8, 'employment_status': 'Employed', 'bankruptcy_in_last_7_years': False, 'late_payments_in_last_2_years': 0, 'credit_inquiries_last_2_years': 1, 'has_existing_loan': False, 'liquid_savings': 75000, 'loan_amount': 25000, 'loan_purpose': 'Home Improvement', 'loan_to_value_ratio': 0.5, 'loan_term_months': 36, 'age': 45, 'is_citizen': True}
Rule Check: Excellent Credit Score
Rule Check: No Recent Bankruptcy
Rule Check: No Late Payments
Rule Check: Very High Income
Rule Check: Stable Employment
Rule Check: Very Low DTI Ratio
Rule Check: No Existing Loans
Rule Check: Large Savings
Rule Check: Home Improvement Purpose
Rule Check: Low Loan-to-Value (Mortgage)
Rule Check: Short Loan Term
Rule Check: Strength: High Income & Low DTI
Rule Check: Strength: Excellent Credit & Stable Job

Final Score Calculation -->
Total Dec